# Face Expression Recognition — face-pretrained backbones, K-fold CV, ensemble → ONNX

This notebook replaces the YOLO11-cls fine-tuning with a pipeline that fixes the real bottlenecks:

| Problem in the YOLO notebook | What this notebook does |
|---|---|
| Training on full photos (eyes/mouth only a few pixels at 224 px) | **Every image is face-cropped with YuNet** — the same `FaceCropper` the app uses live |
| ImageNet backbone that has never seen expressions | **HSEmotion EfficientNet** pretrained on AffectNet (450k faces); its head is initialised from the matching AffectNet emotions, so it works *before* training. Falls back to ConvNeXt-V2 automatically |
| 15–25 near-identical offline copies per face → memorisation | Originals only + strong **online** augmentation, mixup, label smoothing, EMA |
| One 70/15/15 split → ±6-point noise | Fixed **test set** + **5-fold CV** on the rest; the 5 fold models are **ensembled** |
| Mislabelled images | **Out-of-fold mislabel finder** → `mislabel_candidates.csv` + contact sheet → `label_fixes.csv` |
| Duplicate photos across merged datasets | md5 + perceptual-hash **near-duplicate grouping** (never split across train/test) |
| Mostly-grayscale data with some colour photos | Auto-detected; everything is converted to gray (inside the ONNX model too) |

**Output:** one `fer_ensemble.onnx` (normalisation, grayscale, softmax and flip-TTA inside) + `deployment_metadata.json`, installed into `face_expression_model/` where the Django app picks it up.

**How to use:** run top to bottom. To try another backbone, change `BACKBONE` in the configuration cell and run again from section 5 — crops and splits are cached and shared, and section 9 can ensemble all backbones you trained.

## 0. Environment

In [ ]:
# Installs only what is missing. PyTorch itself is NOT installed here: install the CUDA build first, e.g.
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128     (RTX 50-series needs cu128+)
import sys, subprocess, importlib
from importlib.metadata import version, PackageNotFoundError

NEEDED = {"timm": "timm>=1.0.9", "onnx": "onnx", "onnxruntime": None, "sklearn": "scikit-learn",
          "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn", "cv2": "opencv-python",
          "PIL": "pillow"}
missing = []
for mod, pkg in NEEDED.items():
    if mod == "onnxruntime":
        try:
            import onnxruntime  # noqa
        except ImportError:
            missing.append("onnxruntime-gpu")
        continue
    if importlib.util.find_spec(mod) is None:
        missing.append(pkg)
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no CUDA GPU in this kernel - training will be very slow on the CPU.\n"
          "Install the CUDA build: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128")

In [ ]:
import os, sys, json, time, shutil, random, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# fer_lib.py / fer_infer_onnx.py live next to this notebook
NB_DIR = Path.cwd()
if not (NB_DIR / "fer_lib.py").exists() and (NB_DIR / "training" / "fer_lib.py").exists():
    NB_DIR = NB_DIR / "training"
sys.path.insert(0, str(NB_DIR))
import fer_lib as fl
from fer_infer_onnx import FaceExpressionRecognizer

sns.set_theme(style="whitegrid")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | {p.total_memory / 1024**3:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}")
import timm; print("timm", timm.__version__)

## 1. Configuration

* `SOURCE_DIR` — one sub-folder per class. Class names are taken from the folder names (`CLASS_NAMES = None`).
  Files named `<name>__aug<k>.*` are recognised as offline-augmented copies and are **not** used by default.
* `BACKBONE` — one of `hsemotion_b0` (fast, recommended first), `hsemotion_b2`, `convnextv2_tiny`, `effnetv2_s`, `dinov2_vits14`, `convnext_small`.
* `FACE_MARGIN` — must be the same in training and in the app; it is stored in `deployment_metadata.json`.

In [ ]:
# ---------------- Paths ----------------
REPO_ROOT = NB_DIR.parent if (NB_DIR.parent / "manage.py").exists() else NB_DIR
SOURCE_DIR = REPO_ROOT / "Data"                       # e.g. Path(r"D:\Shitol\augmented_image") for the 6-class set
DETECTOR_PATH = REPO_ROOT / "face_expression_model" / "face_detection_yunet_2023mar.onnx"
WORK_DIR = REPO_ROOT / "work" / "fer_crops"           # face crops (cache, safe to delete)
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "fer_kfold" # one sub-folder per backbone + export/
APP_MODEL_DIR = REPO_ROOT / "face_expression_model"   # Django app model folder
LABEL_FIXES_CSV = REPO_ROOT / "label_fixes.csv"       # optional, see section 7
WEIGHTS_CACHE = REPO_ROOT / "work" / "weights"        # HSEmotion downloads

CLASS_NAMES = None            # None = sub-folders of SOURCE_DIR, e.g. ["Angry", "Fear", "Happy", "Sad", "Suprise"]

# ---------------- Data ----------------
FACE_MARGIN = 0.20            # crop = face box + 20% on every side (square)
ASSUME_CROP_BELOW = 100       # images smaller than this (e.g. 48x48) are already face crops
CROP_SIZE = 320               # stored crop size (bigger crops are downscaled, small ones kept as is)
OFFLINE_COPIES_PER_ORIGINAL = 0   # >0 also trains on k "__aug" copies per original (train folds only)
GRAYSCALE = "auto"            # "auto": True if >50% of the originals are gray; or True / False
NEAR_DUP_HAMMING = 3          # perceptual-hash distance for "same picture" (0 disables near-dup merging)
TEST_FRACTION = 0.15
N_FOLDS = 5
FOLDS_TO_RUN = None           # None = all folds; e.g. [0] for a quick first experiment
SEED = 42

# ---------------- Model / training ----------------
BACKBONE = "hsemotion_b0"
PRETRAINED = True
EPOCHS = "auto"               # "auto": 30 for small datasets, 12 for > 20k training images
PATIENCE = 8
TRAIN_OVERRIDES = {}          # e.g. {"lr_backbone": 5e-5, "batch": 32, "mixup_alpha": 0.0}
WORKERS = min(8, os.cpu_count() or 2)
RETRAIN = False               # False = skip folds that already have best.pt (resume after a crash)

# ---------------- Deployment ----------------
CONF_THRESHOLD = 0.50         # below this the app may show "uncertain"
EXPORT_BACKBONES = "best"     # "best" = best OOF combination of trained backbones; or e.g. ["hsemotion_b0"]
INSTALL_TO_APP = True         # copy fer_ensemble.onnx + metadata into face_expression_model/

# ---------------- Smoke test (used to check the code end-to-end on a tiny subset) ----------------
SMOKE_TEST = os.environ.get("FER_SMOKE_TEST", "") == "1"
SMOKE_PER_CLASS = int(os.environ.get("FER_SMOKE_PER_CLASS", "40"))
if SMOKE_TEST:
    PRETRAINED = os.environ.get("FER_SMOKE_PRETRAINED", "0") == "1"
    EPOCHS, PATIENCE, N_FOLDS, WORKERS = 2, 2, 3, 0
    WORK_DIR, ARTIFACT_ROOT = WORK_DIR.parent / "fer_crops_smoke", ARTIFACT_ROOT.parent / "fer_kfold_smoke"
    APP_MODEL_DIR = ARTIFACT_ROOT / "app_model_dir"
    TRAIN_OVERRIDES = {"batch": 16}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_DIR = ARTIFACT_ROOT / BACKBONE
for d in (WORK_DIR, RUN_DIR, WEIGHTS_CACHE):
    d.mkdir(parents=True, exist_ok=True)
if not SOURCE_DIR.exists():
    raise FileNotFoundError(SOURCE_DIR)
if not DETECTOR_PATH.exists():
    raise FileNotFoundError(f"{DETECTOR_PATH} (YuNet face detector) not found")
print("Source:", SOURCE_DIR, "| backbone:", BACKBONE, "| device:", DEVICE, "| run dir:", RUN_DIR)

## 2. Index the dataset (originals, duplicates, label fixes)

In [ ]:
t0 = time.perf_counter()
src_df, scan_report = fl.scan_source(SOURCE_DIR, CLASS_NAMES, workers=WORKERS or 4)
CLASS_NAMES = CLASS_NAMES or fl.detect_class_names(SOURCE_DIR)
src_df, fix_report = fl.apply_label_fixes(src_df, LABEL_FIXES_CSV, CLASS_NAMES)

if SMOKE_TEST:   # tiny stratified subset of originals (+ their copies)
    keep_groups = (src_df[~src_df["augmented"]].sample(frac=1.0, random_state=SEED)
                   .groupby("label").head(SMOKE_PER_CLASS)["group"])
    src_df = src_df[src_df["group"].isin(set(keep_groups))].reset_index(drop=True)

counts = (src_df.groupby(["label", "augmented"]).size().unstack(fill_value=0)
          .reindex(CLASS_NAMES, fill_value=0).rename(columns={False: "originals", True: "aug_copies"}))
display(counts)
print(json.dumps({**{k: v for k, v in scan_report.items() if k != "conflicting_files"}, **fix_report}, indent=2))
print(f"Classes: {CLASS_NAMES}  ({time.perf_counter() - t0:.0f}s)")

## 3. Face crops

Every original is face-cropped with YuNet (largest face, square, `FACE_MARGIN`). Images smaller than `ASSUME_CROP_BELOW` px are already crops. Crops are cached in `WORK_DIR`; re-running is instant.
Then near-duplicate pictures (perceptual hash) are grouped so they never land on both sides of a split, and near-duplicates with **different** labels are removed as ambiguous.

In [ ]:
crop_df, crop_report = fl.build_face_crops(
    src_df, WORK_DIR, DETECTOR_PATH, margin=FACE_MARGIN, crop_size=CROP_SIZE,
    assume_crop_below=ASSUME_CROP_BELOW, workers=WORKERS or 4,
    copies_per_original=OFFLINE_COPIES_PER_ORIGINAL, seed=SEED)
print(json.dumps(crop_report, indent=2))

if NEAR_DUP_HAMMING > 0:
    crop_df, dup_report = fl.merge_near_duplicates(crop_df, NEAR_DUP_HAMMING)
    dup_conflicts = dup_report.pop("_conflicts")
    dup_conflicts.to_csv(RUN_DIR.parent / "near_duplicate_label_conflicts.csv", index=False)
    print(json.dumps(dup_report, indent=2))

orig_df = crop_df[~crop_df["augmented"]]
if GRAYSCALE == "auto":
    GRAYSCALE = bool(orig_df["gray"].mean() > 0.5)
print(f"Grayscale originals: {orig_df['gray'].mean() * 100:.1f}% -> GRAYSCALE = {GRAYSCALE}")
no_face = orig_df[orig_df["crop_mode"] == "no_face"]
no_face[["path", "label"]].to_csv(RUN_DIR.parent / "no_face_detected.csv", index=False)
print(f"Photos where YuNet found no face (centre square used): {len(no_face)} -> no_face_detected.csv "
      "(check them: often not a usable face -> drop via label_fixes.csv)")

In [ ]:
# Preview: random crops per class (top) and photos where YuNet found no face (bottom row)
cols = 10
fig, axes = plt.subplots(len(CLASS_NAMES) + 1, cols, figsize=(1.6 * cols, 1.8 * (len(CLASS_NAMES) + 1)))
for r, c in enumerate(CLASS_NAMES):
    sub = orig_df[orig_df["label"] == c].sample(min(cols, (orig_df["label"] == c).sum()), random_state=SEED)
    for k in range(cols):
        ax = axes[r, k]; ax.axis("off")
        if k < len(sub):
            ax.imshow(Image.open(sub.iloc[k]["crop_path"]).convert("RGB"))
            if k == 0: ax.set_title(c, fontsize=10, loc="left")
nf = no_face.sample(min(cols, len(no_face)), random_state=SEED) if len(no_face) else orig_df.iloc[:0]
for k in range(cols):
    ax = axes[-1, k]; ax.axis("off")
    if k < len(nf):
        ax.imshow(Image.open(nf.iloc[k]["crop_path"]).convert("RGB"))
        if k == 0: ax.set_title("no face detected", fontsize=10, loc="left", color="red")
plt.tight_layout(); plt.savefig(RUN_DIR.parent / "crop_preview.png", dpi=120); plt.show()

## 4. Leakage-free split: fixed test set + K folds (by picture group)

In [ ]:
split_df = fl.make_splits(crop_df, CLASS_NAMES, TEST_FRACTION, N_FOLDS, SEED)
test_df = split_df[(split_df["split"] == "test") & ~split_df["augmented"]].reset_index(drop=True)
tv_orig = split_df[(split_df["split"] == "trainval") & ~split_df["augmented"]].reset_index(drop=True)
tv_aug = split_df[(split_df["split"] == "trainval") & split_df["augmented"]]

tab = pd.crosstab(split_df.loc[~split_df["augmented"], "label"],
                  split_df.loc[~split_df["augmented"], "fold"].map(lambda f: "test" if f < 0 else f"fold{f}"))
tab["total"] = tab.sum(axis=1)
display(tab.reindex(CLASS_NAMES))

# A signature of the exact split, so that runs of different backbones can be ensembled safely
SPLIT_SIG = fl.hashlib.md5(("|".join(tv_orig["crop_path"]) + "#" + "|".join(test_df["crop_path"])
                            + "#" + "|".join(map(str, tv_orig["fold"]))).encode()).hexdigest()[:12]
split_df.to_csv(RUN_DIR.parent / "split.csv", index=False)
print("Split signature:", SPLIT_SIG, "| trainval originals:", len(tv_orig), "| test originals:", len(test_df))

n_train_est = len(tv_orig) * (N_FOLDS - 1) / N_FOLDS + len(tv_aug)
if EPOCHS == "auto":
    EPOCHS = 30 if n_train_est < 20000 else 12
print("Epochs per fold:", EPOCHS)

## 5. Backbone check

For HSEmotion the 6 (or 5) output rows start as the corresponding AffectNet emotions, so the model is evaluated **zero-shot** first: well above chance (1/num_classes) proves the weights loaded correctly. If HSEmotion cannot be downloaded/loaded, the code falls back to ConvNeXt-V2 automatically.

In [ ]:
model, SPEC = fl.create_model(BACKBONE, CLASS_NAMES, WEIGHTS_CACHE, pretrained=PRETRAINED)
SPEC["grayscale"] = GRAYSCALE
print({k: v for k, v in SPEC.items() if k not in ("mean", "std")})
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

if SPEC["resolved"].startswith("hsemotion:"):
    probe = tv_orig.sample(frac=1.0, random_state=SEED).groupby("label").head(300)
    p0 = fl.predict_paths(model, SPEC, probe["crop_path"].tolist(), DEVICE, tta=False, batch=128)
    y0 = probe["label"].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
    zs = accuracy_score(y0, p0.argmax(1))
    print(f"Zero-shot accuracy on {len(probe)} training faces: {zs:.3f} (chance = {1 / len(CLASS_NAMES):.3f})")
    if zs < 1.5 / len(CLASS_NAMES) and not SMOKE_TEST:
        print("WARNING: zero-shot accuracy is near chance - check that the class folders are named "
              "after emotions, or set BACKBONE='convnextv2_tiny'.")
del model

## 6. Train the K folds

In [ ]:
spec_defaults = {k: SPEC[k] for k in ("batch", "lr_backbone", "lr_head", "freeze_epochs") if k in SPEC}
CFG = fl.TrainConfig(epochs=EPOCHS, patience=PATIENCE, workers=WORKERS, seed=SEED, **spec_defaults)
for k, v in TRAIN_OVERRIDES.items():
    setattr(CFG, k, v)
print(CFG)

cid = {c: i for i, c in enumerate(CLASS_NAMES)}
y_tv = tv_orig["label"].map(cid).to_numpy()
oof_path, info_path = RUN_DIR / "oof_probs.npy", RUN_DIR / "run_info.json"
same_split = info_path.exists() and json.loads(info_path.read_text()).get("split_sig") == SPLIT_SIG
if oof_path.exists() and same_split and not RETRAIN:
    oof = np.load(oof_path)                      # resume
else:
    if not same_split and any(RUN_DIR.glob("fold*/best.pt")):
        print("Split changed since the last run -> retraining all folds of", BACKBONE)
        for d in RUN_DIR.glob("fold*"):
            shutil.rmtree(d)
    oof = np.full((len(tv_orig), len(CLASS_NAMES)), np.nan, dtype=np.float32)

fold_rows = []
folds = FOLDS_TO_RUN if FOLDS_TO_RUN is not None else list(range(N_FOLDS))
for k in folds:
    fold_dir = RUN_DIR / f"fold{k}"
    va_mask = (tv_orig["fold"] == k).to_numpy()
    if (fold_dir / "best.pt").exists() and not RETRAIN and not np.isnan(oof[va_mask]).any():
        h = pd.read_csv(fold_dir / "history.csv")
        print(f"fold {k}: already trained (best val acc {h['val_acc'].max():.4f}) - skipped")
        fold_rows.append({"fold": k, "val_acc": h["val_acc"].max(), "best_epoch": int(h.loc[h["val_acc"].idxmax(), "epoch"]),
                          "epochs_run": len(h)})
        continue
    train_df = tv_orig[~va_mask]
    if len(tv_aug):
        train_df = pd.concat([train_df, tv_aug[tv_aug["fold"] != k]])
    val_df = tv_orig[va_mask]
    print(f"\n=== fold {k}: train {len(train_df)} | val {len(val_df)} ===")
    model, spec = fl.create_model(BACKBONE, CLASS_NAMES, WEIGHTS_CACHE, pretrained=PRETRAINED, verbose=False)
    spec["grayscale"] = GRAYSCALE
    t0 = time.perf_counter()
    res = fl.train_one_fold(model, spec, train_df, val_df, CLASS_NAMES, CFG, fold_dir, DEVICE, log_every=1)
    oof[va_mask] = res["val_probs"]
    np.save(oof_path, oof)
    (RUN_DIR / "run_info.json").write_text(json.dumps({"split_sig": SPLIT_SIG, "backbone": BACKBONE,
                                                       "classes": CLASS_NAMES, "spec": {k2: (list(v) if isinstance(v, tuple) else v) for k2, v in spec.items()}}, indent=2))
    fold_rows.append({"fold": k, "val_acc": res["val_acc"], "best_epoch": res["best_epoch"],
                      "epochs_run": len(res["history"]), "minutes": round((time.perf_counter() - t0) / 60, 1)})
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_rows)
display(fold_df)

## 7. Cross-validation results and mislabel finder

In [ ]:
done = ~np.isnan(oof).any(1)
oof_pred = oof[done].argmax(1)
oof_acc = accuracy_score(y_tv[done], oof_pred)
lo, hi = fl.bootstrap_ci(y_tv[done], oof_pred)
print(f"Out-of-fold accuracy on {done.sum()} trainval faces: {oof_acc:.4f}  (95% CI {lo:.3f}-{hi:.3f})")
print(f"Out-of-fold macro-F1: {f1_score(y_tv[done], oof_pred, average='macro'):.4f}\n")
print(classification_report(y_tv[done], oof_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for k in folds:
    h = pd.read_csv(RUN_DIR / f"fold{k}" / "history.csv")
    axes[0].plot(h["epoch"], h["train_loss"], "--", alpha=.6, label=f"fold{k} train")
    axes[0].plot(h["epoch"], h["val_loss"], label=f"fold{k} val")
    axes[1].plot(h["epoch"], h["val_acc"], label=f"fold{k}")
axes[0].set(title="Loss", xlabel="epoch"); axes[1].set(title="Validation accuracy", xlabel="epoch")
axes[0].legend(fontsize=7, ncol=2); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(RUN_DIR / "training_curves.png", dpi=150); plt.show()

### Fix labels (usually the biggest single gain)

`mislabel_candidates.csv` lists trainval faces whose label the out-of-fold model disagrees with most strongly; `mislabel_candidates.jpg` shows them. Open the sheet, then in the CSV fill `action` with **drop** (unusable / ambiguous) or **relabel** (+ `new_label`), save it as `label_fixes.csv` in the repo root and re-run the notebook. Fixes are applied in section 2; your image folders are never modified.

In [ ]:
cand = fl.mislabel_candidates(tv_orig[done].reset_index(drop=True), oof[done], CLASS_NAMES,
                              top=300 if len(tv_orig) > 10000 else 150)
cand.to_csv(RUN_DIR / "mislabel_candidates.csv", index=False)
fl.contact_sheet(cand.head(96), RUN_DIR / "mislabel_candidates.jpg", cols=12, cell=120,
                 title_fn=lambda r: [f"{r.Index}: {r.label}->{r.predicted}", f"p={r.p_predicted:.2f}"])
print(f"{len(cand)} candidates -> {RUN_DIR / 'mislabel_candidates.csv'}")
display(cand[["file", "label", "predicted", "p_label", "p_predicted"]].head(15))
if (RUN_DIR / "mislabel_candidates.jpg").exists():
    plt.figure(figsize=(18, 12)); plt.imshow(Image.open(RUN_DIR / "mislabel_candidates.jpg")); plt.axis("off"); plt.show()

## 8. Test set: ensemble of the fold models

In [ ]:
fold_models = []
for k in folds:
    m, spec_k, ck = fl.load_fold_model(RUN_DIR / f"fold{k}" / "best.pt", device=DEVICE)
    fold_models.append((m, spec_k))

# TTA decision on out-of-fold data (never on test)
plain = np.zeros_like(oof); tta_p = np.zeros_like(oof)
for k, (m, s) in zip(folds, fold_models):
    va = (tv_orig["fold"] == k).to_numpy()
    paths = tv_orig.loc[va, "crop_path"].tolist()
    plain[va] = fl.predict_paths(m, s, paths, DEVICE, tta=False)
    tta_p[va] = fl.predict_paths(m, s, paths, DEVICE, tta=True)
acc_plain = accuracy_score(y_tv[done], plain[done].argmax(1)); acc_tta = accuracy_score(y_tv[done], tta_p[done].argmax(1))
USE_TTA = bool(acc_tta > acc_plain)
print(f"OOF accuracy plain {acc_plain:.4f} | flip-TTA {acc_tta:.4f} -> USE_TTA = {USE_TTA}")

y_test = test_df["label"].map(cid).to_numpy()
test_paths = test_df["crop_path"].tolist()
per_fold = [fl.predict_paths(m, s, test_paths, DEVICE, tta=USE_TTA) for m, s in fold_models]
test_probs = np.mean(per_fold, axis=0)
np.save(RUN_DIR / "test_probs.npy", test_probs)
np.save(RUN_DIR / "oof_probs_final.npy", tta_p if USE_TTA else plain)

y_pred = test_probs.argmax(1)
test_acc = accuracy_score(y_test, y_pred)
lo, hi = fl.bootstrap_ci(y_test, y_pred)
print("Single fold models on test:", [round(accuracy_score(y_test, p.argmax(1)), 4) for p in per_fold])
print(f"ENSEMBLE test accuracy: {test_acc:.4f}  (95% CI {lo:.3f}-{hi:.3f}, n={len(y_test)})")
print(f"ENSEMBLE test macro-F1: {f1_score(y_test, y_pred, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, digits=4, zero_division=0))

metrics = {"backbone": BACKBONE, "resolved": SPEC["resolved"], "split_sig": SPLIT_SIG, "oof_acc": oof_acc,
           "test_acc": test_acc, "test_ci95": [lo, hi], "test_macro_f1": f1_score(y_test, y_pred, average="macro"),
           "tta": USE_TTA, "folds": folds, "n_test": int(len(y_test)), "n_trainval": int(len(tv_orig))}
(RUN_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

cm = confusion_matrix(y_test, y_pred, labels=range(len(CLASS_NAMES)))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0], cmap="Blues")
sns.heatmap(cm / cm.sum(1, keepdims=True).clip(1), annot=True, fmt=".2f", xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES, ax=axes[1], cmap="Blues", vmin=0, vmax=1)
axes[0].set(title="Test confusion (counts)", xlabel="Predicted", ylabel="Actual")
axes[1].set(title="Test confusion (recall)", xlabel="Predicted", ylabel="Actual")
plt.tight_layout(); plt.savefig(RUN_DIR / "test_confusion.png", dpi=150); plt.show()

In [ ]:
# Confidence threshold: accuracy on the answered faces vs. how many are answered
conf = test_probs.max(1); correct = y_pred == y_test
rows = [{"threshold": t, "coverage": (conf >= t).mean(), "accuracy_answered": correct[conf >= t].mean() if (conf >= t).any() else np.nan}
        for t in np.round(np.arange(0.3, 0.96, 0.05), 2)]
thr = pd.DataFrame(rows); thr.to_csv(RUN_DIR / "threshold_tradeoff.csv", index=False)
display(thr.style.format({"coverage": "{:.1%}", "accuracy_answered": "{:.1%}"}))

## 9. Ensemble across backbones (optional)

Train more than one `BACKBONE` (change it in section 1 and re-run sections 5–8). This cell finds every trained backbone with the **same split**, greedily picks the combination with the best **out-of-fold** accuracy (test is only reported, never used to choose), and that combination is exported in section 10 when `EXPORT_BACKBONES = "best"`.

In [ ]:
runs = {}
for d in sorted(ARTIFACT_ROOT.iterdir()):
    f = d / "metrics.json"
    if f.exists() and (d / "oof_probs_final.npy").exists():
        m = json.loads(f.read_text())
        if m.get("split_sig") == SPLIT_SIG and len(m.get("folds", [])) == N_FOLDS:
            runs[d.name] = {"oof": np.load(d / "oof_probs_final.npy"), "test": np.load(d / "test_probs.npy"), **m}
if not runs:   # e.g. only some folds trained so far
    runs[BACKBONE] = {"oof": (tta_p if USE_TTA else plain), "test": test_probs, **metrics}
display(pd.DataFrame([{"backbone": k, "oof_acc": v["oof_acc"], "test_acc": v["test_acc"]} for k, v in runs.items()]))

mask = done
chosen, best_acc = [], -1
remaining = list(runs)
while remaining:
    scores = {r: accuracy_score(y_tv[mask], np.mean([runs[c]["oof"][mask] for c in chosen + [r]], 0).argmax(1)) for r in remaining}
    r, a = max(scores.items(), key=lambda kv: kv[1])
    if a <= best_acc:
        break
    chosen.append(r); remaining.remove(r); best_acc = a
combo_test = np.mean([runs[c]["test"] for c in chosen], 0)
print(f"Best combination by OOF accuracy: {chosen} -> OOF {best_acc:.4f} | "
      f"TEST {accuracy_score(y_test, combo_test.argmax(1)):.4f}")
BEST_COMBO = chosen

## 10. Export one ONNX model and install it in the app

All fold models of the chosen backbone(s) are wrapped into **one** ONNX graph: input `float32 N×3×S×S` RGB in [0, 1] → output class **probabilities** (grayscale conversion, per-model resize/normalisation, flip-TTA and averaging are inside). The export is then checked with the deployment module (`fer_infer_onnx.py`) on the test crops.

In [ ]:
export_list = BEST_COMBO if EXPORT_BACKBONES == "best" else list(EXPORT_BACKBONES)
members = []
for b in export_list:
    for k in range(N_FOLDS):
        ck = ARTIFACT_ROOT / b / f"fold{k}" / "best.pt"
        if ck.exists():
            m, s, _ = fl.load_fold_model(ck, device="cpu")
            members.append((m.float().eval(), s))
input_size = max(int(s["img_size"]) for _, s in members)
wrapper = fl.EnsembleWrapper(members, input_size=input_size, tta=USE_TTA)

EXPORT_DIR = ARTIFACT_ROOT / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
onnx_path = EXPORT_DIR / "fer_ensemble.onnx"
max_diff = fl.export_onnx(wrapper, onnx_path)
print(f"Exported {len(members)} models ({export_list}) -> {onnx_path} "
      f"({onnx_path.stat().st_size / 1e6:.1f} MB), max |torch - onnx| = {max_diff:.2e}")

meta = {
    "task": "face_expression_classification",
    "format_version": 2,
    "onnx_file": onnx_path.name,
    "classes": CLASS_NAMES,
    "num_classes": len(CLASS_NAMES),
    "image_size": input_size,
    "input": "float32 [N,3,S,S] RGB in [0,1]; face crop resized so the short side is S, centre crop",
    "output": "probabilities",
    "grayscale_inside_model": bool(GRAYSCALE),
    "tta_inside_model": USE_TTA,
    "face_detector": DETECTOR_PATH.name,
    "face_margin": FACE_MARGIN,
    "face_crop": "square, side = max(w,h) * (1 + 2*margin), edge-replicate padding (fer_infer_onnx.square_crop)",
    "assume_crop_below": ASSUME_CROP_BELOW,
    "confidence_threshold": CONF_THRESHOLD,
    "members": [{"backbone": s["backbone"], "resolved": s["resolved"], "img_size": s["img_size"]} for _, s in members],
    "metrics": {"oof_acc": float(best_acc), "test_acc": float(accuracy_score(y_test, combo_test.argmax(1))),
                "n_test": int(len(y_test))},
    "split_sig": SPLIT_SIG,
    "created": time.strftime("%Y-%m-%d %H:%M"),
    "torch": torch.__version__, "timm": timm.__version__, "python": platform.python_version(),
}
(EXPORT_DIR / "deployment_metadata.json").write_text(json.dumps(meta, indent=2))
shutil.copy2(DETECTOR_PATH, EXPORT_DIR / DETECTOR_PATH.name)
shutil.copy2(NB_DIR / "fer_infer_onnx.py", EXPORT_DIR / "fer_infer_onnx.py")

# Verify: deployment module (onnxruntime, its own preprocessing) vs. PyTorch on the test crops
rec = FaceExpressionRecognizer(EXPORT_DIR)
onnx_probs = np.concatenate([rec.classify_faces([fl.read_bgr(p) for p in test_paths[i:i + 64]])
                             for i in range(0, len(test_paths), 64)])
check = {"providers": rec.session.get_providers(),
         "test_acc_onnx": float(accuracy_score(y_test, onnx_probs.argmax(1))),
         "test_acc_torch": float(accuracy_score(y_test, combo_test.argmax(1))),
         "top1_agreement": float((onnx_probs.argmax(1) == combo_test.argmax(1)).mean())}
print(json.dumps(check, indent=2))
(EXPORT_DIR / "onnx_check.json").write_text(json.dumps(check, indent=2))
if check["top1_agreement"] < 0.98:
    print("WARNING: ONNX and PyTorch disagree on >2% of test faces - check preprocessing.")

In [ ]:
if INSTALL_TO_APP:
    APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for f in ("fer_ensemble.onnx", "deployment_metadata.json"):
        shutil.copy2(EXPORT_DIR / f, APP_MODEL_DIR / f)
    if not (APP_MODEL_DIR / DETECTOR_PATH.name).exists():
        shutil.copy2(DETECTOR_PATH, APP_MODEL_DIR / DETECTOR_PATH.name)
    print("Installed into", APP_MODEL_DIR, "- restart the Django server to load the new model.")

## 11. Latency (ONNX Runtime, classifier only)

In [ ]:
crops = [fl.read_bgr(p) for p in test_paths[:64]]
def bench(fn, n):
    fn(); t = []
    for _ in range(n):
        s = time.perf_counter(); fn(); t.append((time.perf_counter() - s) * 1000)
    return float(np.median(t))
one = bench(lambda: rec.classify_faces(crops[:1]), 30)
batch = bench(lambda: rec.classify_faces(crops), 10)
print(json.dumps({"providers": rec.session.get_providers(), "single_face_ms": round(one, 2),
                  "batch64_ms": round(batch, 2), "faces_per_s_batched": round(64000 / batch)}, indent=2))

## 12. What next, if accuracy still isn't enough

1. **Fix labels** with `mislabel_candidates.csv` → `label_fixes.csv` (section 7), then re-run. Typically the largest gain.
2. **Train a second backbone** (`hsemotion_b2` or `convnextv2_tiny`) and let section 9 ensemble them (+1–3%).
3. **Bigger input** for HSEmotion-B2 / ConvNeXt: `TRAIN_OVERRIDES = {"batch": 32}` and `BACKBONES[...]["img_size"]`.
4. **Hard pairs** (sad↔neutral, fear↔surprise): collect more *real* faces of those classes; augmented copies do not add information.
5. Only after the above: tune `lr_backbone` (×0.5 / ×2), `mixup_alpha` (0 / 0.4), `label_smoothing` (0.05 / 0.15) — compare on **OOF** accuracy, not test.